# Week 2 — Small Model Training (Run A Baseline)

**Project:** Can We Trust AI-Image Detectors on Ugandan Content?
**Member:** AW — change `MEMBER` below if needed.
**Goal:** Train a small CNN from scratch on a tiny balanced subset, save the model and metrics.

### RAM-safe design
- Images: 128×128 (tiny, fits in RAM)
- Batch: 8, Epochs: 5
- No Hugging Face streaming — loads from local pilot images
- Run cells **one at a time**, never "Run all"

### Progressive scaling plan
This notebook uses `N_IMAGES = 200`. To scale up later, just change `N_IMAGES` and `EPOCHS` — the code stays identical.

### Drive paths
- Root: `/content/drive/MyDrive/Deepfake Detection Project/datasets`
- Pilot images: `pilot_data/real/`, `pilot_data/ai/`
- Outputs: `reports/`

In [ ]:
!pip -q install torch torchvision scikit-learn pandas pillow

In [ ]:
import os, sys, json, glob, hashlib, time, gc
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image as PILImage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix, classification_report,
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print('PyTorch:', torch.__version__)
print('Imports OK')

## 1. Mount Drive + Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ===========================================================================
# CURRENT DRIVE ROOT (matches Week 1 notebook)
# ===========================================================================
ROOT = '/content/drive/MyDrive/Deepfake Detection Project/datasets'
MEMBER = 'AW'

# --- PROGRESSIVE CONFIG: change these to scale up later ---
N_IMAGES = 200          # total images (100 real + 100 AI)
IMG_SIZE = 128          # input resolution
BATCH_SIZE = 8          # small batch for RAM safety
EPOCHS = 5              # training epochs
LR = 1e-3               # learning rate
SEED = 42               # fixed seed for reproducibility
SPLIT_TEST = 0.2        # 20% held out for testing

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Output folders
for d in ['reports', 'metadata', 'audit_outputs']:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)

print('Drive root:', ROOT)
print('Config: %d images, %dx%d, batch %d, %d epochs' % (N_IMAGES, IMG_SIZE, IMG_SIZE, BATCH_SIZE, EPOCHS))

## 2. Load pilot images from Week 1

Loads images saved in `pilot_data/real/` and `pilot_data/ai/` from the Week 1 acquisition notebook.
If you haven't run Week 1 yet, this cell will create a tiny synthetic set so the pipeline still runs.

In [ ]:
real_dir = os.path.join(ROOT, 'pilot_data', 'real')
ai_dir = os.path.join(ROOT, 'pilot_data', 'ai')

def load_pilot_images(real_dir, ai_dir, n_each, img_size):
    paths, labels, meta = [], [], []

    # Real images
    real_files = sorted(glob.glob(os.path.join(real_dir, '*.jpg'))) + \
                 sorted(glob.glob(os.path.join(real_dir, '*.png')))
    for p in real_files[:n_each]:
        paths.append(p); labels.append(0)
        meta.append({'path': p, 'label': 'real', 'source': 'pilot'})

    # AI images
    ai_files = sorted(glob.glob(os.path.join(ai_dir, '*.jpg'))) + \
               sorted(glob.glob(os.path.join(ai_dir, '*.png')))
    for p in ai_files[:n_each]:
        paths.append(p); labels.append(1)
        meta.append({'path': p, 'label': 'ai', 'source': 'pilot'})

    print(f'Found {len(real_files)} real, {len(ai_files)} AI files on Drive')
    print(f'Loading {len(paths)} images ({len([l for l in labels if l==0])} real, {len([l for l in labels if l==1])} AI)')

    if len(paths) == 0:
        print('No pilot images found — creating tiny synthetic set so pipeline runs.')
        for i in range(n_each):
            # Real: noise blob
            arr = np.random.randint(0, 255, (img_size, img_size, 3), dtype=np.uint8)
            im = PILImage.fromarray(arr)
            p = os.path.join(real_dir, f'synthetic_real_{i:03d}.jpg')
            im.save(p, quality=95)
            paths.append(p); labels.append(0)
            meta.append({'path': p, 'label': 'real', 'source': 'synthetic'})
            # AI: gradient blob
            arr = np.zeros((img_size, img_size, 3), dtype=np.uint8)
            for x in range(img_size):
                arr[:, x, :] = [int(255 * x / img_size), 128, 64]
            im = PILImage.fromarray(arr)
            p = os.path.join(ai_dir, f'synthetic_ai_{i:03d}.jpg')
            im.save(p, quality=95)
            paths.append(p); labels.append(1)
            meta.append({'path': p, 'label': 'ai', 'source': 'synthetic'})

    return paths, labels, meta

n_each = N_IMAGES // 2
paths, labels, meta_list = load_pilot_images(real_dir, ai_dir, n_each, IMG_SIZE)

## 3. Convert to tensors + train/test split

In [ ]:
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225])

X_list, y_list = [], []
for path, label in zip(paths, labels):
    im = PILImage.open(path).convert('RGB')
    t = normalize(to_tensor(im))
    X_list.append(t)
    y_list.append(float(label))
    im.close()

X = torch.stack(X_list)
y = torch.tensor(y_list, dtype=torch.float32)
del X_list, y_list
gc.collect()

X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X, y, meta_list, test_size=SPLIT_TEST, random_state=SEED, stratify=y.int().numpy())

train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=torch.cuda.is_available())

print(f'train: {len(train_ds)} | test: {len(test_ds)}')
print(f'train real={int((y_train==0).sum())} ai={int((y_train==1).sum())}')
print(f'test  real={int((y_test==0).sum())} ai={int((y_test==1).sum())}')

## 4. Define TinyCNN (Run A baseline)

In [ ]:
class TinyCNN(nn.Module):
    """Small CNN trained from scratch. Same architecture used across weeks."""
    def __init__(self, img_size=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),  # 64
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),  # 32
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),  # 16
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x.squeeze(1)


model = TinyCNN(img_size=IMG_SIZE).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print('Parameters:', sum(p.numel() for p in model.parameters()))
print(model)

## 5. Train (5 epochs)

In [ ]:
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    train_loss = total_loss / max(n, 1)

    model.eval()
    with torch.no_grad():
        probs = []
        for xb, _ in train_loader:
            xb = xb.to(DEVICE)
            probs.append(torch.sigmoid(model(xb)).cpu().numpy())
        train_probs = np.concatenate(probs)
    train_preds = (train_probs >= 0.5).astype(int)
    train_acc = accuracy_score(y_train.int().numpy(), train_preds)

    print(f'Epoch {epoch:2d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f}')
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc})

gc.collect()
print('Training complete.')

## 6. Evaluate on test set

In [ ]:
model.eval()
y_true, y_prob = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        y_prob.extend(probs.tolist())
        y_true.extend(yb.numpy().tolist())

y_true = np.array(y_true).astype(int)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
try:
    auc = roc_auc_score(y_true, y_prob)
except ValueError:
    auc = float('nan')
cm = confusion_matrix(y_true, y_pred)

print('=== WEEK 2 TEST RESULTS ===')
print(f'Accuracy : {acc:.4f}')
print(f'Macro-F1 : {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'ROC AUC  : {auc:.4f}')
print('Confusion matrix (rows=true 0=real,1=ai; cols=pred):')
print(cm)

# Save model + metrics
model_path = os.path.join(ROOT, 'reports', 'tinycnn_baseline.pt')
torch.save(model.state_dict(), model_path)

metrics = {
    'week': 2,
    'member': MEMBER,
    'model': 'TinyCNN (from scratch)',
    'n_train': len(train_ds),
    'n_test': len(test_ds),
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'accuracy': float(acc),
    'macro_f1': float(f1),
    'precision': float(prec),
    'recall': float(rec),
    'roc_auc': float(auc),
    'confusion_matrix': cm.tolist(),
    'history': history,
    'timestamp': datetime.utcnow().isoformat() + 'Z',
}
metrics_path = os.path.join(ROOT, 'reports', 'week2_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\nModel saved to {model_path}')
print(f'Metrics saved to {metrics_path}')
gc.collect()